# DAY2 실습 02 — 비율·교차표·피벗표

In [33]:
%pip install -q koreanize-matplotlib

import numpy as np
import pandas as pd
import koreanize_matplotlib

In [34]:
import os

from google.colab import drive

# Google Drive를 Colab에 연결한다. 실행하면 권한 승인 창이 나타난다 → "Google Drive에 연결" → 계정 선택 → "허용"
drive.mount("/content/drive")
data_folder = "/content/drive/MyDrive/HD_AX_S/data"
csv_path = os.path.join(data_folder, "superstore_1.csv")

df = pd.read_csv(csv_path)

data = df.copy()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 데이터와 분석 단위

01과 같은 superstore.csv를 사용한다. 원자료의 한 행은 상품행이므로, 01처럼 주문 단위로 묶은 `orders`를 만든 뒤 분석한다. `orders`의 한 행은 주문 1건이다.

이 자료에서는 **주문이익이 0보다 작은 주문을 적자 주문**으로 정한다. 상품행 하나가 손해여도 같은 주문의 다른 상품에서 이익이 나면 주문 전체는 적자가 아닐 수 있다. 그래서 적자 여부는 상품행이 아니라 주문 단위로 판단한다.

| 열 | 뜻 |
|---|---|
| order_id | 주문번호 |
| region | 지역 : Central · East · South · West |
| segment | 고객군 : Consumer · Corporate · Home Office |
| 주문매출 | 주문에 들어 있는 상품행 매출의 합 |
| 주문이익 | 주문에 들어 있는 상품행 이익의 합 (손해면 음수) |

공식 문서 : [이름 붙여 집계하기 (agg)](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation)

In [35]:
# [예제] 01과 같은 주문 단위 표 만들기
# 같은 주문번호의 상품행을 묶어 매출·이익은 더하고(sum), 지역·고객군은 하나만 가져온다(first)

orders = data.groupby("order_id", as_index=False).agg(
    region=("region", "first"),
    segment=("segment", "first"),
    주문매출=("sales", "sum"),
    주문이익=("profit", "sum")
)

print("주문 수:", len(orders), "고유 주문 수:", orders["order_id"].nunique())
display(orders.head())

주문 수: 5111 고유 주문 수: 5111


,order_id,region,segment,주문매출,주문이익
0,CA-2023-100867,West,Home Office,32.700,8.5000
1,CA-2023-107153,East,Corporate,10.688,3.7408
2,CA-2023-115238,East,Corporate,281.160,89.2000
3,CA-2023-115777,East,Consumer,387.990,182.3600
4,CA-2023-117964,East,Corporate,56.424,-18.5354


## 1. 비율의 분자와 분모

**비율**은 관심 있는 사건의 건수(분자)를 기준이 되는 전체 건수(분모)로 나눈 값이다. 백분율(%)은 비율에 100을 곱한 값이다.

- 비율 = 분자 ÷ 분모
- 백분율(%) = 비율 × 100

### 0/1 변수의 평균은 사건 비율이다

사건이 일어나면 1, 일어나지 않으면 0으로 표시한 변수를 **0/1 변수**라고 한다. 0/1 변수를 더하면 1의 개수(사건 건수)가 되고, 평균을 내면 1의 개수 ÷ 유효 관측수, 즉 **사건 비율**이 된다.

예) 0, 1, 1, 0, 1 → 합 3 (사건 3건), 평균 3 ÷ 5 = 0.6 (사건 비율 60%)

### 건수와 비율

| 지표 | 답하는 질문 | 예 |
|---|---|---|
| 건수 | 사건이 얼마나 많이 일어났는가 (규모) | 이익이 적자인 주문이 가장 많은 지역은? |
| 비율 | 집단 안에서 사건이 얼마나 자주 일어났는가 (발생 정도) | 주문 중 이익이 적자인 주문의 비율이 가장 높은 지역은? |

주문이 많은 지역은 적자 건수도 많아지기 쉽다. 집단 크기가 다르면 건수만으로 비교하지 않고 비율을 함께 본다.

| 코드 | 의미 |
|---|---|
| `orders["주문이익"] < 0` | 주문마다 적자이면 True, 아니면 False |
| `astype(int)` | True → 1, False → 0 으로 바꿔 0/1 변수를 만든다 |

공식 문서 : [astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) · [mean](https://pandas.pydata.org/docs/reference/api/pandas.Series.mean.html)

In [36]:
# [실습]
# 상황 : 지역별로 적자 주문이 얼마나 많이 나왔는지(규모) 먼저 확인한다.
# 지역별 전체 주문 수와 적자 주문 수는 각각 몇 건인가?
# 주문이익이 0보다 작으면 1, 아니면 0인 적자여부 열을 orders 에 만들고, 결과 표는 cnt 라는 이름에 담는다 (뒤 예제와 문제에서 사용한다)

orders["적자여부"] = (orders["주문이익"] < 0).astype(int)

cnt = orders.groupby("region")["적자여부"].agg(
    전체주문="size",
    적자주문="sum"
)
display(cnt)

,전체주문,적자주문
region,,
Central,1179,415
East,1475,326
South,822,140
West,1635,159


### 직접 해보기

지역별 적자 주문 비율(%)을 두 가지 방법으로 구하고 같은지 확인한다. 하나는 적자 주문 수 ÷ 전체 주문 수, 다른 하나는 적자여부(0/1)의 평균이다.

In [37]:
# [실습]
# 상황 : 이익이 적자인 주문이 잦은 지역을 찾아 그 지역의 운영 방식을 점검하려 한다.
#        지역마다 주문 수가 다르므로 '그 지역 주문 중' 이익이 적자인 주문이 얼마나 잦은지 비교한다.
# 지역별 적자 주문 비율(%)을 '적자 주문 수 ÷ 전체 주문 수 × 100'과 '적자여부의 평균 × 100'으로 각각 구하시오.
# 결과 표는 rate 라는 이름에 담는다

rate = cnt.copy()
rate["적자주문비율(%)"] = rate["적자주문"] / rate["전체주문"] * 100
rate["평균으로 구한 비율(%)"] = orders.groupby("region")["적자여부"].mean() * 100
display(rate.round(2))

,전체주문,적자주문,적자주문비율(%),평균으로 구한 비율(%)
region,,,,
Central,1179,415,35.20,35.20
East,1475,326,22.10,22.10
South,822,140,17.03,17.03
West,1635,159,9.72,9.72


## 2. 집단 내부 비율과 집단 구성비

같은 분자(예 : Central의 적자 주문 수)라도 무엇을 분모로 두느냐에 따라 다른 질문에 답한다.

| 구분 | 집단 내부 비율 | 집단 구성비 |
|---|---|---|
| 질문 | 이 집단 안에서 사건이 몇 %인가? | 전체 사건 중 이 집단이 몇 %를 차지하는가? |
| 분자 | 그 집단의 사건 건수 | 그 집단의 사건 건수 |
| 분모 | 그 집단의 전체 건수 | 모든 집단의 사건 건수 |
| 예 | Central 주문 중 이익이 적자인 주문의 비율 | 이익이 적자인 주문 전체 중 Central의 몫 |
| 모든 집단을 더하면 | 100%가 되지 않는다 | 100%가 된다 |

집단 내부 비율은 **어느 집단에서 사건이 잦은가**를, 집단 구성비는 **사건이 어느 집단에 몰려 있는가**를 보여 준다. 비율을 보고할 때는 무엇을 분모로 했는지 함께 적는다.

In [38]:
# [실습]
# 상황 : 회사 전체의 적자 주문 감축 목표를 지역별로 나누려 한다. '적자 주문 전체 중' 각 지역이 차지하는 몫을 구하고,
#        적자가 잦은 지역(앞 문제)과 적자가 몰려 있는 지역이 같은지 확인한다.
# rate 에 '전체 적자 중 지역 구성비(%)' 열을 추가하고, 적자주문비율(%)과 비교하시오.

rate["전체 적자 중 지역 구성비(%)"] = rate["적자주문"] / rate["적자주문"].sum() * 100
display(rate[["적자주문", "적자주문비율(%)", "전체 적자 중 지역 구성비(%)"]].round(2))

print("적자주문비율 합계(%):", round(rate["적자주문비율(%)"].sum(), 2))
print("지역 구성비 합계(%):", round(rate["전체 적자 중 지역 구성비(%)"].sum(), 2))

,적자주문,적자주문비율(%),전체 적자 중 지역 구성비(%)
region,,,
Central,415,35.20,39.90
East,326,22.10,31.35
South,140,17.03,13.46
West,159,9.72,15.29


적자주문비율 합계(%): 84.06
지역 구성비 합계(%): 100.0


## 3. 결측이 있을 때의 분모

사건 여부에 결측값(NaN)이 있으면 **전체 행 수**와 **유효 관측수**가 달라지고, 어느 쪽을 분모로 쓰느냐에 따라 비율의 뜻이 달라진다.

| 분모 | 계산 | 비율의 뜻 |
|---|---|---|
| 전체 행 수 | `groupby("집단").size()` | 전체 기록 중 사건이 확인된 비율 |
| 유효 관측수 | `groupby("집단")["열"].count()` | 사건 여부를 아는 기록 중 사건 비율 |

- 둘 중 하나가 틀린 것이 아니다. 분석 목적에 맞는 분모를 고르고, 보고할 때 어느 기준인지 밝힌다.
- `mean()`은 결측을 빼고 계산하므로 유효 관측수 기준 비율이 된다.

공식 문서 : [count](https://pandas.pydata.org/docs/reference/api/pandas.Series.count.html)

In [39]:
# [예제] 전체 행 수와 유효 관측수로 나눈 비율 비교
# A 집단에는 발생 여부를 모르는 기록(NaN)이 하나 있다

example = pd.DataFrame({"집단": ["A", "A", "A", "B", "B"], "발생": [1, 0, np.nan, 1, 0]})

n_all = example.groupby("집단").size()               # 결측 포함 전체 행 수
n_valid = example.groupby("집단")["발생"].count()     # 결측을 뺀 유효 관측수
n_event = example.groupby("집단")["발생"].sum()       # 발생(1) 건수

missing_table = pd.DataFrame({"전체 행 수": n_all, "유효관측수": n_valid, "결측수": n_all - n_valid, "확인된 발생 수": n_event})
missing_table["전체 행 대비 확인된 발생(%)"] = n_event / n_all * 100
missing_table["유효관측 중 발생(%)"] = n_event / n_valid * 100
display(missing_table)

,전체 행 수,유효관측수,결측수,확인된 발생 수,전체 행 대비 확인된 발생(%),유효관측 중 발생(%)
집단,,,,,,
A,3,2,1,1.0,33.333333,50.0
B,2,2,0,1.0,50.000000,50.0


## 4. 교차표

**교차표**(crosstab)는 두 범주형 변수의 값 조합마다 건수를 세어 행과 열로 정리한 표다. 한 변수의 값을 행에, 다른 변수의 값을 열에 놓고, 각 칸에 그 조합의 건수를 적는다.

| 구성 | 예 : 지역 × 적자여부 |
|---|---|
| 행 | 지역 (region) |
| 열 | 적자여부 (0 = 적자 아님, 1 = 적자) |
| 칸 | 그 지역에서 적자여부가 그 값인 주문 수 |
| 합계 (All) | 행·열마다 더한 값. `margins=True`로 붙인다 |

`orders`는 한 행이 주문 1건이므로 교차표의 칸은 주문 수다. 상품행 자료(`data`)로 만들면 칸이 상품행 수가 되므로, 교차표를 만들기 전에도 한 행이 무엇인지 먼저 확인한다.

기본 형식 : `pd.crosstab(행으로 둘 열, 열로 둘 열)`


- 행 : 비교하려는 **집단** (그룹 기준) — 예 : 지역
- 열 : 집단마다 확인할 **결과(사건)** — 예 : 적자여부

공식 문서 : [crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [40]:
# [예제] 지역 × 고객군 주문 수 교차표

# 1) 조합별 주문 수
ct = pd.crosstab(orders["region"], orders["segment"])
display(ct)

# 2) 합계 행·열 붙이기 : margins=True
display(pd.crosstab(orders["region"], orders["segment"], margins=True))

segment,Consumer,Corporate,Home Office
region,,,
Central,608,348,223
East,741,468,266
South,444,247,131
West,835,489,311


segment,Consumer,Corporate,Home Office,All
region,,,,
Central,608,348,223,1179
East,741,468,266,1475
South,444,247,131,822
West,835,489,311,1635
All,2628,1552,931,5111


In [41]:
# [실습]
# 상황 : 1절에서 groupby 로 구한 지역별 적자 주문 수를 교차표 한 장으로 정리한다.
# 지역 × 적자여부 주문 수 교차표를 합계(margins=True)와 함께 만드시오.



## 5. 교차표 비율 세 가지

교차표의 같은 칸도 **무엇을 분모로 두느냐**에 따라 비율이 세 가지로 나온다. `normalize` 옵션으로 분모를 정한다.

| 구분 | 집단 내부 비율 (행 비율) | 집단 구성비 (열 비율) | 전체 비율 |
|---|---|---|---|
| `normalize` | `"index"` | `"columns"` | `"all"` |
| 분모 | 그 행의 합계 | 그 열의 합계 | 표 전체 합계 |
| 질문 (지역 × 적자여부) | 이 지역 주문 중 적자는 몇 %인가? | 적자 주문 중 이 지역은 몇 %인가? | 전체 주문 중 이 지역·적자 조합은 몇 %인가? |
| 100%가 되는 방향 | 각 행 | 각 열 | 표 전체 |

- 결과는 0 ~ 1 사이 비율이므로 `* 100`을 붙여 백분율로 본다.

### 어떤 비율을 쓸지 고르는 법

계산하기 전에 질문을 **"[분모] 중 [분자]는 몇 %인가?"** 문장으로 바꿔 본다. '~ 중', '~ 안에서' 자리에 오는 것이 분모다.

1) 분모가 **행 변수의 한 값**(이 지역)이면 `"index"`, **열 변수의 한 값**(적자 주문)이면 `"columns"`, **전체**면 `"all"`
2) 행에 비교할 집단, 열에 결과를 두면, 집단끼리 결과 비율을 비교하는 질문은 항상 `"index"`다

| 결과활용| 질문을 문장으로 바꾸면 | 분모 | normalize |
|---|---|---|---|
| 문제가 **잦은 집단**을 찾아 원인을 점검한다 | 각 지역 주문 **중** 적자는 몇 %인가? | 그 지역 주문 | `"index"` |
| 사건을 줄이려고 **어디에 몰려 있는지** 보고 인력·목표를 나눈다 | 적자 주문 **중** 각 지역은 몇 %인가? | 적자 주문 전체 | `"columns"` |
| 전체에서 **가장 큰 몫**인 조합을 찾아 우선 관리한다 | 전체 주문 **중** 이 조합은 몇 %인가? | 전체 주문 | `"all"` |



공식 문서 : [crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html)

In [42]:
# [예제] 행 비율 : 지역 안에서 적자 주문 비율
# normalize="index" : 각 행(지역)의 합계를 분모로 쓴다

row_fail = pd.crosstab(orders["region"], orders["적자여부"], normalize="index") * 100
display(row_fail.round(2))

적자여부,0,1
region,,
Central,64.80,35.20
East,77.90,22.10
South,82.97,17.03
West,90.28,9.72


In [43]:
# [실습]전체 적자 중 지역 구성비
# 지역 × 적자여부 교차표를 열 비율(normalize="columns")로 구하고, 2절의 구성비와 같은지 확인하시오.

fail_ct = pd.crosstab(orders["region"], orders["적자여부"], margins=True)
fail_ct

적자여부,0,1,All
region,,,
Central,764,415,1179
East,1149,326,1475
South,682,140,822
West,1476,159,1635
All,4071,1040,5111


### 직접 해보기

지역 × 고객군 교차표(`ct`)로 세 가지 비율을 차례로 구하고, Central·Consumer 칸 하나가 각각 무엇을 뜻하는지 확인한다.

In [44]:
# [실습]
# 상황 : 지역 주문 중 어느 고객군의 비중이 큰지 보고 홍보 대상을 정하려 한다.
# 1. 각 지역의 전체 주문 중 고객군별 주문 비율(행 비율)은 몇 %인가?
# 2. 각 지역의 고객군 비율 합계가 100%인지 확인하시오.
# 3. Central 지역의 Consumer 비율을 직접 계산하여 비교하시오.



# 1. 지역별 고객군 주문 비율(%)
#    normalize="index" : 각 행(지역)의 합계를 기준으로 비율 계산
#    * 100 : 비율을 백분율로 변환

ct_rate = pd.crosstab(
    orders["region"],
    orders["segment"],
    normalize="index"
) * 100

display(ct_rate.round(2))


# 2. 지역별 비율 합계 확인
#    axis=1 : 각 행에서 열 방향으로 합산
#    to_frame() : Series를 DataFrame으로 변환하고 열 이름 지정

print("지역별 구성비 합계(%)")

display(
    ct_rate.sum(axis=1).to_frame(name="구성비 합계(%)")
)


# 3. Central 지역의 Consumer 비율 직접 계산
#    ct : 앞에서 만든 주문 건수 교차표
#    loc[행이름, 열이름] : 교차표에서 칸 하나를 꺼낸다
#    ct.loc["Central", "Consumer"] : Central의 Consumer 주문 수
#    ct.loc["Central"].sum() : Central 지역 전체 주문 수

central_rate = (
    ct.loc["Central", "Consumer"]
    / ct.loc["Central"].sum() * 100
)

print("Central Consumer 직접 계산(%):", round(central_rate, 2))
print("crosstab 구성비(%):", round(ct_rate.loc["Central", "Consumer"], 2))   # 이미 계산된 비율 가져오기

segment,Consumer,Corporate,Home Office
region,,,
Central,51.57,29.52,18.91
East,50.24,31.73,18.03
South,54.01,30.05,15.94
West,51.07,29.91,19.02


지역별 구성비 합계(%)


,구성비 합계(%)
region,
Central,100.0
East,100.0
South,100.0
West,100.0


Central Consumer 직접 계산(%): 51.57
crosstab 구성비(%): 51.57


In [45]:
# [실습]
# 상황 : 고객군의 주문이 어느 지역에 몰려 있는지 보고 영업 인력을 배치하려 한다.
# 1. 각 고객군의 전체 주문 중 지역별 주문 구성비(열 비율)는 몇 %인가?
# 2. Consumer 전체 주문 중 Central 주문의 구성비를 직접 계산하시오.



# 1. 고객군별 지역 구성비(%)
#    normalize="columns" : 각 열(고객군)의 합계를 기준으로 비율 계산
#    * 100 : 백분율로 변환

col_rate = pd.crosstab(
    orders["region"],
    orders["segment"],
    normalize="columns"
) * 100

display(col_rate.round(2))


# 2. Central 지역의 Consumer 구성비 직접 계산
#    ct : 앞에서 만든 주문 건수 교차표
#    분자 : Central 지역의 Consumer 주문 수
#    분모 : 모든 지역의 Consumer 주문 수

central_rate = (
    ct.loc["Central", "Consumer"]
    / ct["Consumer"].sum() * 100
)

print("Central Consumer 직접 계산(%):", round(central_rate, 2))


# 3. crosstab에서 계산한 구성비와 비교
print("crosstab 구성비(%):", round(col_rate.loc["Central", "Consumer"], 2))


# 4. 고객군별 구성비 합계 확인
#    axis=0 : 각 열에서 행 방향으로 합산
print("고객군별 구성비 합계(%)")
display(col_rate.sum(axis=0).to_frame(name="구성비 합계(%)"))

segment,Consumer,Corporate,Home Office
region,,,
Central,23.14,22.42,23.95
East,28.20,30.15,28.57
South,16.89,15.91,14.07
West,31.77,31.51,33.40


Central Consumer 직접 계산(%): 23.14
crosstab 구성비(%): 23.14
고객군별 구성비 합계(%)


,구성비 합계(%)
segment,
Consumer,100.0
Corporate,100.0
Home Office,100.0


In [46]:
# [실습]
# 상황 : 전체 주문에서 가장 큰 몫을 차지하는 지역·고객군 조합을 찾아 우선 관리 대상으로 정하려 한다.
# 1. 전체 주문 중 지역·고객군별 조합 구성비(전체 비율)는 몇 %인가?
# 2. Central·Consumer 주문 구성비를 직접 계산하여 확인하시오.


# 1. 전체 주문 기준 구성비(%)
#    normalize="all" : 교차표 전체 합계를 분모로 사용
all_rate = pd.crosstab(
    orders["region"],
    orders["segment"],
    normalize="all"
) * 100

display(all_rate.round(2))


# 2. Central·Consumer 구성비 직접 계산
#    분자 : Central 지역의 Consumer 주문 수
#    분모 : 전체 주문 수 (orders 의 행 수)
central_all = (
    ct.loc["Central", "Consumer"]
    / len(orders) * 100
)

print("Central Consumer 직접 계산(%):", round(central_all, 2))


# 3. crosstab에서 계산한 값과 비교
print("crosstab 구성비(%):", round(all_rate.loc["Central", "Consumer"], 2))

segment,Consumer,Corporate,Home Office
region,,,
Central,11.90,6.81,4.36
East,14.50,9.16,5.20
South,8.69,4.83,2.56
West,16.34,9.57,6.08


Central Consumer 직접 계산(%): 11.9
crosstab 구성비(%): 11.9


## 6. 피벗표

**피벗표**(pivot table)는 원자료를 지정한 집계 방법으로 요약해 행 × 열 형태로 펼친 표다. `groupby()`에 기준을 두 개 주면 결과가 **긴 표**(조합 하나가 한 행)로 나오고, 피벗표는 같은 결과를 **넓은 표**(한 기준은 행, 다른 기준은 열)로 보여 준다.

### 세 가지 도구를 언제 쓰나

| 도구 | 언제 쓰나 | 칸에 들어가는 값 |
|---|---|---|
| `groupby()` | 집단별로 숫자 열을 요약할 때 | 평균·합계·n (긴 표) |
| `pd.crosstab()` | 범주형 두 개의 조합이 몇 건인지, 비율이 얼마인지 볼 때 | 건수 또는 비율 |
| `pivot_table()` | 요약 결과를 행 × 열로 펼쳐 비교·보고할 때 | 평균·합계 등 집계값 |

| 옵션 | 의미 | 예 |
|---|---|---|
| `index` | 행 기준 | `"region"` |
| `columns` | 열 기준 | `"segment"` |
| `values` | 집계할 열 | `"주문매출"` |
| `aggfunc` | 집계 방법 (기본값은 평균) | `"mean"`, `"count"` 등 |


공식 문서 : [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) · [groupby](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html)

In [47]:
# [예제] 두 기준으로 나눈 groupby 결과 : 긴 표
# 지역 · 고객군 조합 하나가 한 행이 된다

by_two = orders.groupby(["region", "segment"])["주문매출"].mean()
display(by_two)

region   segment    
Central  Consumer       417.701013
         Corporate      454.010956
         Home Office    409.025309
East     Consumer       481.972822
         Corporate      434.992788
         Home Office    492.893541
South    Consumer       440.497682
         Corporate      493.465314
         Home Office    566.832073
West     Consumer       435.897891
         Corporate      475.148803
         Home Office    461.386190
Name: 주문매출, dtype: float64

In [48]:
# [실습]
# 지역을 행, 고객군을 열로 둔 주문당 평균매출 피벗표를 만드시오.

pivot = orders.pivot_table(index="region", columns="segment", values="주문매출", aggfunc="mean")
display(pivot.round(2))

segment,Consumer,Corporate,Home Office
region,,,
Central,417.70,454.01,409.03
East,481.97,434.99,492.89
South,440.50,493.47,566.83
West,435.90,475.15,461.39


### 직접 해보기

평균매출 피벗표와 같은 모양으로 주문 수(n) 피벗표를 만들고, 평균이 가장 높은 칸의 n을 확인한다.

In [49]:
# [실습]
# 상황 : 피벗표에서 '평균매출이 가장 높은 조합'을 보고하려 한다.
# 지역 × 고객군별 주문 수(n)를 피벗표로 구하시오. (aggfunc="count")

pivot_n = orders.pivot_table(index="region", columns="segment", values="주문매출", aggfunc="count")
display(pivot_n)

segment,Consumer,Corporate,Home Office
region,,,
Central,608,348,223
East,741,468,266
South,444,247,131
West,835,489,311


## 7. 정리

| 확인할 것 | 함수·계산 | 답하는 질문 |
|---|---|---|
| 0/1 변수 만들기 | `(조건).astype(int)` | 사건이면 1, 아니면 0 |
| 사건 건수 | 0/1 변수의 `sum()` | 사건이 몇 건인가 (규모) |
| 사건 비율 | 0/1 변수의 `mean()` 또는 건수 ÷ 전체 | 집단 안에서 사건이 몇 %인가 (발생 정도) |
| 교차표 | `pd.crosstab(행, 열)` | 두 범주의 조합이 몇 건인가 |
| 합계 붙이기 | `margins=True` | 행·열·전체 합계는 얼마인가 |
| 집단 내부 비율 | `normalize="index"` | 이 행 안에서 몇 %인가 |
| 집단 구성비 | `normalize="columns"` | 이 열 전체 중 이 행이 몇 %인가 |
| 전체 비율 | `normalize="all"` | 전체 중 이 조합이 몇 %인가 |
| 피벗표 | `pivot_table(index, columns, values, aggfunc)` | 요약값을 행 × 열로 비교하면? |

순서
1) 한 행이 무엇인지 확인한다 (상품행인지 주문인지)
2) 사건을 정의하고 0/1 변수로 만든다
3) 건수와 비율을 함께 본다. 비율은 분모를 먼저 정한다
4) 교차표 비율은 normalize 로 분모를 고른다 (행 · 열 · 전체)
5) 피벗표의 평균은 칸마다 n 을 함께 본다

공식 문서 : [crosstab](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) · [pivot_table](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) · [astype](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html)

## 8. 종합 실습 : East·Corporate 주문의 세 가지 비율

앞에서 만든 교차표와 비율 표를 쓰지 않고 `orders`에서 새로 계산한다. East·Corporate 칸 하나를 세 가지 분모로 읽고, 각각 무엇을 뜻하는지 문장으로 정리한다.

In [50]:
# 문제 1)
# 상황 : East 지역 Corporate 고객의 주문 비중을 보고해야 한다. 먼저 분자와 분모가 될 건수를 모두 구한다.
# 지역·고객군 주문 수 교차표(count_table)를 만들고, East·Corporate 주문 수, East 전체 주문 수, Corporate 전체 주문 수, 전체 주문 수를 구하시오.

count_table = pd.crosstab(orders["region"], orders["segment"])

counts = pd.Series({
    "East·Corporate 주문 수": count_table.loc["East", "Corporate"],
    "East 전체 주문 수": count_table.loc["East"].sum(),
    "Corporate 전체 주문 수": count_table["Corporate"].sum(),
    "전체 주문 수": len(orders),
})
counts

,0
East·Corporate 주문 수,468
East 전체 주문 수,1475
Corporate 전체 주문 수,1552
전체 주문 수,5111


In [51]:
# 문제 2)
# East·Corporate 의 집단 내부 비율(East 안에서), 집단 구성비(Corporate 안에서), 전체 비율을 normalize 로 구하시오. (소수점 둘째 자리)


row_table = pd.crosstab(orders["region"], orders["segment"], normalize="index") * 100
col_table = pd.crosstab(orders["region"], orders["segment"], normalize="columns") * 100
all_table = pd.crosstab(orders["region"], orders["segment"], normalize="all") * 100

pd.Series({
    "East 안의 Corporate 비율(%)": row_table.loc["East", "Corporate"],
    "Corporate 중 East 구성비(%)": col_table.loc["East", "Corporate"],
    "전체 주문 중 East·Corporate(%)": all_table.loc["East", "Corporate"],
}).round(2)

,0
East 안의 Corporate 비율(%),31.73
Corporate 중 East 구성비(%),30.15
전체 주문 중 East·Corporate(%),9.16


In [52]:
# 문제 3)
# 문제 2의 세 비율이 각각 어떤 질문에 답하는지 한 문장씩 쓰고, 분자와 분모를 함께 밝히시오.

# - East 안의 Corporate 비율 31.73% : East 주문 1,475건 중 Corporate 주문이 468건이다 (East 의 고객 구성)
# - Corporate 중 East 구성비 30.15% : Corporate 주문 1,552건 중 East 주문이 468건이다 (Corporate 주문이 어느 지역에 있는지)
# - 전체 중 East·Corporate 9.16% : 전체 주문 5,111건 중 East·Corporate 주문이 468건이다 (전체에서 이 조합의 몫)